# CAP182 SS3

In [21]:
import pandas as pd
import numpy as np


## Dataset loading

Purpose
The client supplied multiple datasets containing hospital admissions, patient demographics, clinical information, medication records, outpatient follow-up activity, claims information and procedure data.

The admissions dataset was selected as the primary dataset because it contains the target variable Readmitted_30_Days, indicating whether a patient was readmitted within 30 days of discharge.

The remaining datasets are linked using either Patient_ID or Admission_ID and provide additional predictive information that may improve model performance.

Dataset Overview
Target Variable:

0 = Not Readmitted Within 30 Days
1 = Readmitted Within 30 Days
Dataset size, class distribution and class imbalance are analysed below.

Why CSV?
CSV files are suitable for machine learning workflows because they provide a structured tabular format that can be efficiently loaded into Pandas DataFrames for preprocessing, feature engineering and modelling.

In [22]:
admissions = pd.read_csv("request1_admissions_discharge.csv")
demographics = pd.read_csv("request2_patient_demographics.csv")
clinical = pd.read_csv("request3_medical_history_clinical.csv")
medications = pd.read_csv("request4_medication_treatment.csv")
followup = pd.read_csv("request5_followup_outpatient.csv")
claims = pd.read_csv("request6_claims_billing.csv")
procedures = pd.read_csv("request7_procedures.csv")

print("Admissions:", admissions.shape)
print("Demographics:", demographics.shape)
print("Clinical:", clinical.shape)
print("Medications:", medications.shape)
print("Follow-up:", followup.shape)
print("Claims:", claims.shape)
print("Procedures:", procedures.shape)

print("\nTarget Distribution")
print(admissions["Readmitted_30_Days"].value_counts())

print("\nTarget Percentage")
print(
    admissions["Readmitted_30_Days"]
    .value_counts(normalize=True)
    * 100
)

Admissions: (9174, 13)
Demographics: (5218, 7)
Clinical: (5218, 10)
Medications: (32989, 6)
Follow-up: (4775, 6)
Claims: (9147, 14)
Procedures: (5689, 7)

Target Distribution
Readmitted_30_Days
No     8181
Yes     993
Name: count, dtype: int64

Target Percentage
Readmitted_30_Days
No     89.175932
Yes    10.824068
Name: proportion, dtype: float64


## Dataset processing

Purpose
Data preprocessing was performed to improve data quality and create a single admission-level dataset suitable for machine learning.

The following preprocessing steps were applied:

Duplicate removal
Standardisation of categorical values
Missing value treatment
Outlier handling
Aggregation of one-to-many datasets
Dataset merging
Class imbalance assessment
Train, validation and test splitting
Missing Values
Missing numerical values are replaced using the median because the median is less sensitive to outliers than the mean.

Missing categorical values are replaced using "Unknown".

Outliers
Extreme BMI values were identified and replaced with missing values before imputation because values below 10 or above 80 are medically unrealistic.

Class Imbalance
Class imbalance was assessed because readmission datasets are commonly imbalanced. This influenced the decision to use class weighting during model training.

Aggregation
Medication, follow-up, claims and procedure datasets contain multiple records per admission.

These tables were aggregated to admission level before merging to create a single row per admission suitable for machine learning and to prevent duplicate records.

In [23]:

#___________________REMOVE DUPLICATES______________

demographics = demographics.drop_duplicates()
clinical = clinical.drop_duplicates()


# ______________STANDARDISE TEXT VALUES__________


demographics["Gender"] = (
    demographics["Gender"]
    .astype(str)
    .str.strip()
    .str.title()
)

clinical["Smoking_Status"] = (
    clinical["Smoking_Status"]
    .astype(str)
    .str.strip()
    .str.title()
)


# ___________HANDLE BMI OUTLIERS__________


clinical.loc[
    (clinical["BMI"] < 10) |
    (clinical["BMI"] > 80),
    "BMI"
] = np.nan


#__________ MEDICATION AGGREGATION_________


med_agg = (
    medications
    .groupby("Admission_ID")
    .agg(
        Medication_Count=("Medication_Name", "count"),
        Average_Daily_Dosage=("Daily_Dosage", "mean"),
        Number_of_Medications=("Number_of_Medications", "max"),
        Treatment_Completed=("Treatment_Completed", "max"),
        Follow_Up_Required=("Follow_Up_Required", "max")
    )
    .reset_index()
)


# ___________FOLLOW-UP AGGREGATION_____________


followup_agg = (
    followup
    .groupby("Admission_ID")
    .agg(
        Attended=("Attended", "max"),
        Days_to_Follow_Up=("Days_to_Follow_Up", "mean"),
        Missed_Appointments=("Missed_Appointments", "max")
    )
    .reset_index()
)


# _____________CLAIMS AGGREGATION_____________


claims_agg = (
    claims
    .groupby("Admission_ID")
    .agg(
        Total_Claim_Value=("Total_Claim_Value", "sum"),
        Patient_Copayment=("Patient_Copayment", "sum"),
        Total_Line_Items=("Total_Line_Items", "sum"),
        Settlement_Amount=("Settlement_Amount", "sum"),
        Query_Flag=("Query_Flag", "max")
    )
    .reset_index()
)


# _____________PROCEDURE AGGREGATION__________


procedure_agg = (
    procedures
    .groupby("Admission_ID")
    .agg(
        Procedure_Count=("Procedure_Code", "count"),
        Total_Theatre_Time=("Theatre_Time_Minutes", "sum")
    )
    .reset_index()
)




Final Dataset Shape: (9174, 43)

Missing Values
Patient_ID                      0
Admission_ID                    0
Admission_Date                  0
Discharge_Date                  0
Length_of_Stay                  0
Hospital_ID                     0
Ward_Type                       0
Admission_Type                  0
Procedure_Code                  0
Procedure_Description           0
Discharge_Status                0
Previous_Admissions             0
Readmitted_30_Days              0
Age                             0
Gender                          0
Province                        0
Medical_Scheme                  0
Membership_Type                 0
Employment_Status               0
Chronic_Diabetes                0
Chronic_Hypertension            0
Chronic_Heart_Disease           0
Chronic_Asthma                  0
Number_of_Chronic_Conditions    0
BMI                             0
Smoking_Status                  0
Allergies                       0
Risk_Score                      0


In [ ]:
# ______________MERGE ALL DATASETS___________


df = admissions.merge(
    demographics,
    on="Patient_ID",
    how="left"
)

df = df.merge(
    clinical,
    on="Patient_ID",
    how="left"
)

df = df.merge(
    med_agg,
    on="Admission_ID",
    how="left"
)

df = df.merge(
    followup_agg,
    on="Admission_ID",
    how="left"
)

df = df.merge(
    claims_agg,
    on="Admission_ID",
    how="left"
)

df = df.merge(
    procedure_agg,
    on="Admission_ID",
    how="left"
)



In [ ]:
# ==========================================================
# HANDLE MISSING VALUES
# ==========================================================

for col in df.select_dtypes(include=np.number):
    df[col] = df[col].fillna(df[col].median())

categorical_cols = df.select_dtypes(
    include=["object", "string"]
).columns

for col in categorical_cols:
    df[col] = df[col].fillna("Unknown")



In [ ]:

# DATA QUALITY CHECKS


print("Final Dataset Shape:", df.shape)

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nClass Distribution")
print(df["Readmitted_30_Days"].value_counts())

print("\nClass Percentage")
print(
    df["Readmitted_30_Days"]
    .value_counts(normalize=True)
    * 100
)